Image representation screening

In [ ]:
from pathlib import Path
import os

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Notebooks/01_dataset_inventory.ipynb").is_file())
os.chdir(root)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from io import BytesIO

print("OpenCV version:", cv2.__version__)

In [ ]:
import boto3
import pandas as pd
from pathlib import PurePosixPath

# AWS configuration
BUCKET_NAME = "amal-stitchingnet-data-903473565738-ap-southeast-2-an"
REGION = "ap-southeast-2"
DATA_PREFIX = "data/"

# Connect to S3 using SageMaker execution role
s3 = boto3.client(
    "s3",
    region_name=REGION
)

print("Connected to S3")

%run Notebooks/00_inventory_helpers.ipynb

(image_df, class_distribution, fabric_order, class_order, combination_counts, stratified_sample) = inventory_views(root / "reports/eda/01_s3_inventory.csv")

In [ ]:
def load_stitching_image(filename):

    match = image_df[
        image_df["file_name"] == filename
    ]

    if len(match) == 0:
        raise ValueError(
            f"Image not found: {filename}"
        )

    key = match.iloc[0]["s3_key"]

    response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    image = Image.open(
        BytesIO(response["Body"].read())
    ).convert("RGB")

    return np.array(image)


samples = {
    "Stain and damage": load_stitching_image(
        "F_07_014.jpg"
    ),

    "Needle mark": load_stitching_image(
        "K_08_011.jpg"
    ),

    "Puckering": load_stitching_image(
        "G_06_013.jpg"
    )
}

In [ ]:
def create_image_variants(rgb):

    # Convert colour spaces
    lab = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2LAB
    )

    gray = cv2.cvtColor(
        rgb,
        cv2.COLOR_RGB2GRAY
    )

    # 1. Original
    original = rgb

    # 2. CLAHE on LAB lightness
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    lab_clahe = lab.copy()

    lab_clahe[:, :, 0] = clahe.apply(
        lab[:, :, 0]
    )

    clahe_rgb = cv2.cvtColor(
        lab_clahe,
        cv2.COLOR_LAB2RGB
    )

    # 3. Unsharp masking
    blurred = cv2.GaussianBlur(
        rgb,
        (0, 0),
        sigmaX=1.0
    )

    sharpened = cv2.addWeighted(
        rgb,
        1.5,
        blurred,
        -0.5,
        0
    )

    # 4. Sobel edge magnitude
    gx = cv2.Sobel(
        gray,
        cv2.CV_32F,
        1, 0,
        ksize=3
    )

    gy = cv2.Sobel(
        gray,
        cv2.CV_32F,
        0, 1,
        ksize=3
    )

    magnitude = cv2.magnitude(gx, gy)

    edges = cv2.normalize(
        magnitude,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    ).astype(np.uint8)

    # 5. LAB A channel
    lab_a = cv2.normalize(
        lab[:, :, 1],
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # 6. LAB B channel
    lab_b = cv2.normalize(
        lab[:, :, 2],
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    return {
        "Original": original,
        "CLAHE - Local contrast": clahe_rgb,
        "Unsharp masking": sharpened,
        "Sobel - Edge map": edges,
        "LAB - A channel": lab_a,
        "LAB - B channel": lab_b
    }

In [ ]:
for class_name, image in samples.items():

    variants = create_image_variants(image)

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 9)
    )

    fig.suptitle(
        f"Enhancement Investigation: {class_name}",
        fontsize=15
    )

    for ax, (method, result) in zip(
        axes.flat,
        variants.items()
    ):

        ax.imshow(
            result,
            cmap="gray" if result.ndim == 2 else None,
            interpolation="nearest"
        )

        ax.set_title(method)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path

screening_sample = (
    image_df
    .groupby(
        ["folder_1", "folder_2"],
        group_keys=False
    )
    .sample(
        n=1,
        random_state=42
    )
    .sort_values(
        ["folder_2", "folder_1"]
    )
    .reset_index(drop=True)
)

assert len(screening_sample) == 121

output_dir = Path(
    "reports/eda/enhancement_experiments"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

screening_sample.to_csv(
    output_dir / "screening_121_images.csv",
    index=False
)

print("Selected images:", len(screening_sample))
print("Fabric categories:",
      screening_sample["folder_1"].nunique())
print("Defect categories:",
      screening_sample["folder_2"].nunique())

display(screening_sample.head(15))